# Uncertainty prompt + temperature 0.1

Standalone notebook: **no imports from this repository** — only standard libraries and Hugging Face.

- System prompt: as in the VUF paper (*hedge if uncertain*).
- Generation: `temperature=0.1`, `do_sample=True`.
- Input: CSV with a `question` column (if absent — uses the first column).
- Output: JSONL in `/content/...` (see the config cell).

### Google Colab

1. Run the cell with **imports and config**.
2. Run the next cell **"Upload CSV"** — a **"Choose Files"** button will appear; select your `test.csv` or any CSV with questions.  
   No path to the repository on Colab disk is needed.
3. Then proceed: prompt, read table, model, generation.

### Running locally (not Colab)

In the upload cell, set the `QUESTIONS_CSV` variable to the path of your CSV on disk.

In [ ]:
# Google Colab: установка пакетов. Локально закомментируйте, если уже установлено.
!pip -q install transformers accelerate sentencepiece pandas tqdm

In [ ]:
import json
from pathlib import Path

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


IN_COLAB = _in_colab()
ROOT = Path.cwd().resolve()

# В Colab сохраняем загрузку и результат в /content — путь не зависит от клона репозитория
WORK = Path("/content") if IN_COLAB else ROOT

UPLOADED_CSV = WORK / "uploaded_questions.csv"
OUT_JSONL = WORK / "uncertainty_prompt_t0.1.jsonl"

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
DEVICE_MAP = "auto"
DTYPE = torch.float16

TEMPERATURE = 0.1
MAX_NEW_TOKENS = 100
TOP_P = 0.9
TOP_K = 50
BATCH_SIZE = 8

QUESTIONS_CSV = None  # задаётся в следующей ячейке (загрузка или путь)

print("IN_COLAB:", IN_COLAB)
print("WORK:", WORK)
print("CSV будет:", UPLOADED_CSV, "(после загрузки)")
print("OUT_JSONL:", OUT_JSONL)

In [ ]:
# --- Загрузка CSV с вопросами (Colab: кнопка «Выбрать файлы») ---

if IN_COLAB:
    from google.colab import files

    print("Нажмите «Выбрать файлы» и загрузите CSV (нужна колонка question или первая колонка = вопрос).")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("Файл не выбран.")
    name = next(iter(uploaded.keys()))
    UPLOADED_CSV.write_bytes(uploaded[name])
    QUESTIONS_CSV = UPLOADED_CSV
    print(f"OK: «{name}» → {QUESTIONS_CSV} ({len(uploaded[name])} bytes)")
else:
    # Локально: укажите путь к CSV (или положите файл questions.csv рядом с ноутбуком)
    QUESTIONS_CSV = ROOT / "questions.csv"
    # QUESTIONS_CSV = Path("/полный/путь/к/вашему/test.csv")
    if not QUESTIONS_CSV.exists():
        raise FileNotFoundError(
            f"Нет файла: {QUESTIONS_CSV}\n"
            "Раскомментируйте строку с Path(...) выше или положите questions.csv рядом с ноутбуком."
        )
    print("Локальный CSV:", QUESTIONS_CSV.resolve())

assert QUESTIONS_CSV.exists(), QUESTIONS_CSV

In [ ]:
# Тот же текст, что в verbal_uncertainty/prompts.py — UNCERTAINTY_QA_PROMPT (без импорта из репо)
UNCERTAINTY_SYSTEM = (
    "Answer the following question using a succinct (at most one sentence) and full answer. "
    "If you are uncertain about your answer to the question, convey this uncertainty linguistically by precisely hedging this answer. "
)

In [ ]:
df = pd.read_csv(QUESTIONS_CSV)
if "question" in df.columns:
    questions = df["question"].astype(str).str.strip().tolist()
else:
    questions = df.iloc[:, 0].astype(str).str.strip().tolist()

print("N questions:", len(questions))
print("Example:", questions[0][:120] if questions else "(empty)")

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    device_map=DEVICE_MAP,
)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.generation_config.pad_token_id = tokenizer.pad_token_id

print("Model loaded.")

In [ ]:
def build_prompt(question: str) -> str:
    messages = [
        {"role": "system", "content": UNCERTAINTY_SYSTEM},
        {"role": "user", "content": f"Question: {question}\nAnswer: "},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def generate_batch(batch_questions: list[str]) -> list[str]:
    prompts = [build_prompt(q) for q in batch_questions]
    enc = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True)
    enc = {k: v.to(model.device) for k, v in enc.items()}

    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            top_k=TOP_K,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    prompt_len = enc["input_ids"].shape[1]
    answers = []
    for seq in out:
        gen_tokens = seq[prompt_len:]
        answers.append(tokenizer.decode(gen_tokens, skip_special_tokens=True).strip())
    return answers


OUT_JSONL.parent.mkdir(parents=True, exist_ok=True)

with OUT_JSONL.open("w", encoding="utf-8") as f:
    for start in tqdm(range(0, len(questions), BATCH_SIZE)):
        batch_qs = questions[start : start + BATCH_SIZE]
        batch_ans = generate_batch(batch_qs)
        for j, (q, ans) in enumerate(zip(batch_qs, batch_ans)):
            row = {
                "idx": start + j,
                "question": q,
                "answer": ans,
                "temperature": TEMPERATURE,
                "prompt": "uncertainty_system",
                "model": MODEL_NAME,
            }
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Done:", OUT_JSONL)